In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json, zipfile
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
from baseline import calculate_resilience_cost
import matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)

# --- Load data ---
df = pd.read_csv("data/train.csv", index_col=0)
test_df = pd.read_csv("data/test.csv",  index_col=0)
cost_matrix_df = pd.read_csv("data/cost_matrix.csv", index_col=0)

display(df)

In [ ]:
display(cost_matrix_df.head())


In [ ]:

# Makes no sense to do it this way, ruins the ordinal structure, let's set it to green->yellow->orange->red
cost_matrix_df = cost_matrix_df.sort_values(by=["0"])
cost_matrix_df = cost_matrix_df[['0', '3', '1', '2']]
display(cost_matrix_df.head())
cost_matrix = cost_matrix_df.values

In [ ]:
# Let's rename just in case

In [ ]:
# Data inspection
display(df.describe())
df.info()


In [ ]:
# Let's change the types for efficiency (maybe it happens already automatically though..)


# is_magnitude_int = all(x.is_integer() for x in df["magnitude"])
# print("Is magnitude actually ints: ", is_magnitude_int)
# # Magnitude is proper float

# is_depth_int = all(x.is_integer() for x in df["depth"])
# print("Is depth actually ints: ", is_depth_int)
# # depth should be uint
# df.depth = df.depth.astype("UInt16")

# is_cdi_int = all(x.is_integer() for x in df["cdi"])
# print("Is cdi actually ints: ", is_cdi_int)
# # cdi should be uint
# df.cdi = df.cdi.astype("UInt8")

# is_mmi_int = all(x.is_integer() for x in df["mmi"])
# print("Is mmi actually ints: ", is_mmi_int)
# # mmi should be int
# df.mmi = df.mmi.astype("UInt8")

# is_sig_int = all(x.is_integer() for x in df["sig"])
# print("Is sig actually ints: ", is_sig_int)
# # sig should be int
# df.sig = df.sig.astype("Int16")

# This actually didn't do anything (also should be applied to test df anyway), it probably gets converted back to floats for computation anyway...

In [ ]:
df.info()

In [ ]:
df.alert = pd.Categorical(df.alert, ["green", "yellow", "orange", "red"], ordered=True)
df.sort_values(by=["alert"], inplace=True)
display(df.alert)
df.reset_index(drop=True, inplace=True)
len(df) - len(df.drop_duplicates())
df.info()
# No dups

In [ ]:

sns.barplot(data=df["alert"], estimator="size") 

In [ ]:

# sns.histplot(data=df, x = "magnitude", hue="alert",)
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="magnitude")


In [ ]:
sns.scatterplot(data=df, x=df.magnitude, y=df.index, hue=df.alert)

In [ ]:
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="depth")

In [ ]:
sns.scatterplot(data=df, x=df.depth, y=df.index, hue=df.alert)
# Some outliers for orange


In [ ]:
outliers = (df.alert == "orange") & (df.depth > 100)
df.drop(df[outliers].index, inplace=True)
print("After drops")
sns.scatterplot(data=df, x=df.depth, y=df.index, hue=df.alert)


In [ ]:
# sns.histplot(data=df["cdi"], hue=df["alert"])
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="cdi")


In [ ]:
plt.figure(figsize = (8, 10))

sns.scatterplot(data=df, x=df.cdi, y=df.index, hue=df.alert)


In [ ]:
# sns.histplot(data=df["mmi"])

sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="mmi")


In [ ]:
plt.figure(figsize = (10, 8))

sns.scatterplot(data=df, x=df.mmi, y=df.index, hue=df.alert)
# Looks like some outliers for everything except red

In [ ]:
# outliers = ((df.alert == "orange") & (df.mmi == 9)) | ((df.alert == "yellow") & (df.mmi == 9)) | ((df.alert == "green") & (df.mmi < 3)) # This one seems tricky on the performance
# df.drop(df[outliers].index, inplace=True)
sns.scatterplot(data=df, x=df.mmi, y=df.index, hue=df.alert)


In [ ]:

# sns.histplot(data=df["sig"])
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="sig")

In [ ]:
plt.figure(figsize = (10, 8))
sns.scatterplot(data=df, x=df.sig, y=df.index, hue=df.alert)


In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder ,OrdinalEncoder
categories=[["green", "yellow", "orange", "red"]]
color_to_int = {"green": 0, "yellow": 1, "orange": 2, "red": 3}
# --- Encode target and split ---
# enc = OrdinalEncoder(categories=categories, dtype=int, )
# enc = LabelEncoder()
print("Enc: ")
# display(enc)
# enc.fit(df.loc[:, ["alert"]])
print("Pre mapping y:")
display(df.alert)
y = df.alert.map(color_to_int)
print("Post mapping y:")
display(y)
X = df.drop("alert", axis=1)
X_final = test_df
display(X.corr())
display(df.info())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

pipe = make_pipeline(StandardScaler())
numeric_features = ["magnitude", "depth", "cdi", "mmi", "sig"]
numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler())]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        # ("cat", categorical_transformer, categorical_features),
    ]
)


In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.compose import TransformedTargetRegressor

class_weight = {
        0: 0.075,
        1: 0.35,
        2: 0.5,
        3: 0.125,
    }
class_weight = {
        0: 0.025,
        1: 0.075,
        2: 0.30,
        3: 0.6,
    }
clf = Pipeline(
    # steps=[("preprocessor", preprocessor), ("classifier", SVC(C=700, class_weight="balanced"))])
    steps=[("preprocessor", preprocessor), ("classifier", SVC(C=700, class_weight=class_weight))]
    # steps=[("preprocessor", preprocessor), ("classifier", LinearRegression())]
)
clf

In [ ]:
int_to_color = {0: "green", 1: "yellow", 2: "orange", 3: "red"}
def resilience_scorer(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    rci = calculate_resilience_cost(cm, cost_matrix)
    return rci

In [ ]:
from matplotlib import pyplot as plt
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold

# We use resilience_score rather than accuracy, since this is what will be competed upon
resilience_score = make_scorer(resilience_scorer, greater_is_better=False, )



# The class weights are very interesting, they seem to make a big impact since recall can be slightly controlled 

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)


outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

param_grid = {
    'classifier__gamma': ["scale"],
    'classifier__kernel': [ "rbf"],
    'classifier__C': np.arange(10, 100, 2)
    }
    # 'classifier_class_weigths_': [{"}
    # 'classifier__C': np.linspace(1, 1)}

grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    scoring=resilience_score,
    cv=inner_cv)

nested_scores = cross_val_score(grid_search, X, y, cv=outer_cv, scoring=resilience_score, )
# y


In [ ]:

print(f"Outer CV Scores (Generalization Error): {nested_scores}")
print(f"Mean Nested Stratified CV Resilience Estimate: {np.mean(nested_scores):.4f}")

# Plotting the nested CV scores
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(nested_scores) + 1), nested_scores, color='purple')
plt.axhline(np.mean(nested_scores), color='red', linestyle='--', label=f'Mean Nested Resilience Score ({np.mean(nested_scores):.4f})')
plt.title('Nested Stratified CV Scores (Tuning Error)')
plt.xlabel('Outer Fold Number')
plt.ylabel('Resilience Score')
plt.ylim(-0, -5000)
plt.legend()
plt.show()


In [ ]:
grid_search.fit(X, y)
display(grid_search.best_params_)
display(grid_search.best_score_)
clf = grid_search



In [ ]:
# # --- Scale features ---
# scaler = MinMaxScaler().fit(X_train)
# X_train, X_val, X_test = (
#     scaler.transform(X_train),
#     scaler.transform(X_val),
#     scaler.transform(test_df),
# )

In [ ]:

# # # --- Train baseline model ---
# clf = KNeighborsClassifier(n_neighbors=10)
# clf.fit(X_train, y_train)
# clf


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay


# --- Validate ---
y_val_pred = clf.predict(X_val)
f1 = f1_score(y_val, y_val_pred, average="macro")
cm = confusion_matrix(y_val, y_val_pred)
rci = calculate_resilience_cost(cm, cost_matrix)

print(classification_report(y_val, y_val_pred))

cm_display = ConfusionMatrixDisplay(cm, display_labels=["green", "yellow", "orange", "red"]).plot()
print(f"F1 (macro): {f1:.3f}")
# print("Confusion matrix:\n", cm)
print(f"Resilience Cost: {rci:.2f}")

In [ ]:
from baseline import create_submission

best_params = grid_search.best_params_
final_clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC(
            C=best_params["classifier__C"],
            gamma=best_params["classifier__gamma"],
            kernel=best_params["classifier__kernel"],
            class_weight=class_weight
        ))
    ]
)

final_clf.fit(X, y)
y_final = final_clf.predict(X_final)
display(best_params)
create_submission([int_to_color[i] for i in y_final])